# RBF with Ellipsoid Constraint — Implicit Surface Fitting Demo
## Li et al., Computer Graphics Forum 2004

Demonstrates: synthetic non-ellipsoidal point cloud generation,
RBF implicit surface fitting, and interactive inline visualisation.

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from rbf_ellipsoid_constraint import (
    fit_rbf_ellipsoid_linear,
    evaluate_model_linear,
    generate_ellipsoid_points,
    generate_torus_points,
    generate_superquadric_points,
    generate_bumpy_sphere_points,
    generate_saddle_points,
    generate_synthetic_points,
    load_point_cloud,
)
print("Imports OK")

## 1. Data selection

Set `DATA_FILENAME` to the **filename** (not the full path) of any file in the
`data/` folder (e.g. `"Tibia.csv"`), or leave it as `None` to use a
synthetic point cloud chosen by `SYNTHETIC_SHAPE`.

Supported file formats: `.csv`, `.xyz`, `.obj`, `.ply`, `.m`, `.npy`, `.npz`

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set to a filename in the data/ folder, e.g. "Tibia.csv", or None for synthetic data
DATA_FILENAME = None

# Used only when DATA_FILENAME is None.
# Choices: "torus", "superquadric", "bumpy_sphere", "saddle", "ellipsoid"
SYNTHETIC_SHAPE = "bumpy_sphere"

N_SYNTHETIC_POINTS = 400      # number of points for synthetic data
NOISE_STD          = 0.05     # noise level for synthetic data
SEED               = 42
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
data_dir = os.path.join(os.getcwd(), "..", "data")

if DATA_FILENAME is not None:
    filepath = os.path.join(data_dir, DATA_FILENAME)
    pts = load_point_cloud(filepath)
    print(f"Loaded '{DATA_FILENAME}':  {pts.shape[0]} points")
else:
    pts = generate_synthetic_points(
        shape=SYNTHETIC_SHAPE,
        n_points=N_SYNTHETIC_POINTS,
        noise_std=NOISE_STD,
        seed=SEED,
    )
    print(f"Generated synthetic '{SYNTHETIC_SHAPE}' point cloud: {pts.shape[0]} points")

print(f"x ∈ [{pts[:,0].min():.3f}, {pts[:,0].max():.3f}]")
print(f"y ∈ [{pts[:,1].min():.3f}, {pts[:,1].max():.3f}]")
print(f"z ∈ [{pts[:,2].min():.3f}, {pts[:,2].max():.3f}]")

## 2. Visualise raw point cloud

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=4, alpha=0.5, c=pts[:,2], cmap='viridis')
title = DATA_FILENAME if DATA_FILENAME else f"Synthetic: {SYNTHETIC_SHAPE}"
ax.set_title(f"Input point cloud — {title}")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
plt.tight_layout()
plt.show()

## 3. RBF implicit surface fitting

In [ ]:
result = fit_rbf_ellipsoid_linear(pts, smooth=1e-4)
if result is None:
    print("Fitting failed — no valid eigenvalue found.")
else:
    alpha, beta, centroid, scale = result
    norm_pts = (pts - centroid) / scale
    residuals = evaluate_model_linear(norm_pts, norm_pts, alpha, beta)
    rms = float(np.sqrt(np.mean(residuals**2)))
    print(f"Centroid : {centroid.round(4)}")
    print(f"Scale    : {scale:.4f}")
    print(f"RMS residual on training points: {rms:.6f}")

## 4. Visualise point cloud overlaid with fitted implicit surface

The fitted surface is visualised by evaluating the RBF implicit function F on a grid
of query points and plotting those with |F| below an adaptive threshold as small
transparent dots (marching-cubes-style scatter), overlaid with the original point cloud.

In [ ]:
if result is not None:
    margin = 0.15
    mins = pts.min(axis=0) - margin * (pts.max(axis=0) - pts.min(axis=0))
    maxs = pts.max(axis=0) + margin * (pts.max(axis=0) - pts.min(axis=0))
    grid_n = 30
    gx = np.linspace(mins[0], maxs[0], grid_n)
    gy = np.linspace(mins[1], maxs[1], grid_n)
    gz = np.linspace(mins[2], maxs[2], grid_n)
    GX, GY, GZ = np.meshgrid(gx, gy, gz)
    grid_pts = np.column_stack([GX.ravel(), GY.ravel(), GZ.ravel()])

    norm_grid = (grid_pts - centroid) / scale
    F_grid = evaluate_model_linear(norm_grid, norm_pts, alpha, beta)

    threshold = np.percentile(np.abs(F_grid), 5)
    surface_mask = np.abs(F_grid) < threshold
    surf_pts = grid_pts[surface_mask]

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(pts[:,0], pts[:,1], pts[:,2],
               s=4, alpha=0.4, color='steelblue', label='Input points')
    if len(surf_pts) > 0:
        ax.scatter(surf_pts[:,0], surf_pts[:,1], surf_pts[:,2],
                   s=6, alpha=0.15, color='tomato', label='Fitted surface (F≈0)')
    title = DATA_FILENAME if DATA_FILENAME else f"Synthetic: {SYNTHETIC_SHAPE}"
    ax.set_title(f"RBF Implicit Surface Fit — {title}\nRMS residual = {rms:.5f}")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

## 5. Showcase: all non-ellipsoidal synthetic shapes

In [ ]:
shapes = ["torus", "superquadric", "bumpy_sphere", "saddle"]
fig, axes = plt.subplots(2, 2, figsize=(13, 10),
                         subplot_kw={'projection': '3d'})
for ax, shape in zip(axes.ravel(), shapes):
    p = generate_synthetic_points(shape=shape, n_points=350, noise_std=0.04, seed=0)
    ax.scatter(p[:,0], p[:,1], p[:,2], s=3, alpha=0.5, c=p[:,2], cmap='plasma')
    ax.set_title(shape.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
plt.suptitle("Non-ellipsoidal synthetic point clouds", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Noise sensitivity study

Fit the RBF model to a bumpy-sphere point cloud at increasing noise levels
and plot the RMS implicit residual vs noise inline.

In [ ]:
noise_levels = np.linspace(0.0, 0.5, 15)
rms_errors = []
for sigma in noise_levels:
    p = generate_bumpy_sphere_points(n_points=300, noise_std=float(sigma), seed=0)
    try:
        res = fit_rbf_ellipsoid_linear(p, smooth=1e-4)
        if res is not None:
            a_, b_, c_, s_ = res
            n_ = (p - c_) / s_
            v_ = evaluate_model_linear(n_, n_, a_, b_)
            rms_errors.append(float(np.sqrt(np.mean(v_**2))))
        else:
            rms_errors.append(np.nan)
    except Exception:
        rms_errors.append(np.nan)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(noise_levels, rms_errors, 'o-', color='tab:blue')
ax.set_xlabel("Noise σ")
ax.set_ylabel("RMS implicit residual")
ax.set_title("Fitting accuracy vs noise (bumpy sphere)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()